# Phase A — Listing-Level Feature Engineering

**Owner:** Member 2 (Mohammed)

Builds the listing-level feature layer that Member 3 will aggregate to neighbourhood KPIs, cluster, fit a price model on, and run the policy simulation against. Member 4's chatbot will later cite numbers that trace back to these features.

## Design

All feature logic lives in `src/features.py` — pure functions, no IO, no globals. This notebook is a thin orchestration layer that:

1. loads Member 1's cleaned listings + monthly metrics for each city,
2. calls `engineer_all_features(...)` to add the Phase A columns,
3. saves the result to `data/processed/{city}/{city}_listings_features.csv`,
4. previews and sanity-checks the new columns.

Member 3 can import the same functions from `src.features` when they extend the pipeline.

## Outputs

- `data/processed/barcelona/barcelona_listings_features.csv`
- `data/processed/london/london_listings_features.csv`

## 1. Imports & path setup

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent.resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import numpy as np

from src.data_io import load_clean_listings, load_monthly_metrics, save_features
from src.features import engineer_all_features

print('Repo root:', REPO_ROOT)
print('pandas', pd.__version__)
print('numpy', np.__version__)

Repo root: /Users/bobbypakenham/KPMG_Airbnb_Capstone
pandas 3.0.3
numpy 2.4.6


## 2. Run pipeline for both cities

In [2]:
CITIES = ['barcelona', 'london']
results = {}

for city in CITIES:
    print(f'\n--- {city.upper()} ---')
    listings = load_clean_listings(city)
    monthly = load_monthly_metrics(city)
    print(f'Loaded listings: {listings.shape}, monthly: {monthly.shape}')

    features = engineer_all_features(listings, monthly, city)
    out_path = save_features(features, city)
    print(f'Saved -> {out_path}')
    print(f'Final shape: {features.shape} ({features.shape[1] - listings.shape[1] - 1} new columns)')
    results[city] = features


--- BARCELONA ---
Loaded listings: (2594, 34), monthly: (88821, 17)


Saved -> /Users/bobbypakenham/KPMG_Airbnb_Capstone/data/processed/barcelona/barcelona_listings_features.csv
Final shape: (2594, 50) (15 new columns)

--- LONDON ---


Loaded listings: (9643, 34), monthly: (306822, 17)


Saved -> /Users/bobbypakenham/KPMG_Airbnb_Capstone/data/processed/london/london_listings_features.csv
Final shape: (9643, 50) (15 new columns)


## 3. New columns added

In [3]:
listings_baseline_cols = set(load_clean_listings('barcelona').columns)
feature_cols = [c for c in results['barcelona'].columns if c not in listings_baseline_cols]
print('New columns from Phase A:')
for c in feature_cols:
    print(f'  - {c}')

New columns from Phase A:
  - city
  - is_active_recent
  - high_intensity_flag
  - occupancy_band
  - price_band
  - revenue_band
  - revenue_per_active_day
  - breach_90_flag
  - breach_60_flag
  - breach_30_flag
  - reside_unregistered_flag
  - geo_key
  - geo_level
  - host_revenue_total
  - host_active_listings
  - host_commercial_tier


## 4. Sanity checks per city

Confirm new flags / bands make sense before handing to Member 3.

In [4]:
def summarise(city: str, df: pd.DataFrame) -> None:
    print(f'\n=== {city.upper()} ({len(df):,} listings) ===')

    print('\nis_active_recent:')
    print(df['is_active_recent'].value_counts(dropna=False))

    print('\nhigh_intensity_flag (ttm_days_booked >= 180):')
    print(df['high_intensity_flag'].value_counts(dropna=False))

    print('\noccupancy_band:')
    print(df['occupancy_band'].value_counts(dropna=False))

    print('\nprice_band:')
    print(df['price_band'].value_counts(dropna=False))

    print('\nBreach flags (entire-home only):')
    eh = df['entire_home_flag'].sum()
    for cap in (90, 60, 30):
        flag = f'breach_{cap}_flag'
        n = df[flag].sum()
        rate = 100 * n / eh if eh else float('nan')
        print(f'  {flag}: {n:,} of {eh:,} entire homes ({rate:.1f}%)')

    print('\nreside_unregistered_flag:', df['reside_unregistered_flag'].sum(),
          f'({100*df["reside_unregistered_flag"].sum()/eh:.1f}% of entire homes)' if eh else '')

    print('\ngeo_level:')
    print(df['geo_level'].value_counts(dropna=False))

    print('\nhost_commercial_tier:')
    print(df['host_commercial_tier'].value_counts(dropna=False))

for city, df in results.items():
    summarise(city, df)


=== BARCELONA (2,594 listings) ===

is_active_recent:
is_active_recent
True     1655
False     939
Name: count, dtype: int64

high_intensity_flag (ttm_days_booked >= 180):
high_intensity_flag
False    2070
True      524
Name: count, dtype: int64

occupancy_band:
occupancy_band
low        1424
unknown     699
med         471
Name: count, dtype: int64

price_band:
price_band
unknown    699
mid        474
budget     474
luxury     474
premium    473
Name: count, dtype: int64

Breach flags (entire-home only):
  breach_90_flag: 642 of 1,539 entire homes (41.7%)
  breach_60_flag: 730 of 1,539 entire homes (47.4%)
  breach_30_flag: 817 of 1,539 entire homes (53.1%)

reside_unregistered_flag: 421 (27.4% of entire homes)

geo_level:
geo_level
subdivision    2354
unknown         240
Name: count, dtype: int64

host_commercial_tier:
host_commercial_tier
casual              1293
multi                699
super_commercial     316
commercial           286
Name: count, dtype: int64

=== LONDON (9,643 

## 5. Stacked combined preview

Member 3 will work on a single concatenated table for cross-city analysis. Sanity-check the union here.

In [5]:
combined = pd.concat([results['barcelona'], results['london']], ignore_index=True)
print('Combined shape:', combined.shape)
print('\nListings per city:')
print(combined['city'].value_counts())
print('\nColumns identical across both cities:', set(results['barcelona'].columns) == set(results['london'].columns))

print('\nBreach summary across both cities:')
summary = (
    combined.groupby('city')[['breach_90_flag', 'breach_60_flag', 'breach_30_flag', 'reside_unregistered_flag']]
    .sum()
    .astype(int)
)
summary

Combined shape: (12237, 50)

Listings per city:
city
london       9643
barcelona    2594
Name: count, dtype: int64

Columns identical across both cities: True

Breach summary across both cities:


,breach_90_flag,breach_60_flag,breach_30_flag,reside_unregistered_flag
city,,,,
barcelona,642,730,817,421
london,1485,1868,2270,2572


In [6]:
# Spot check: top 10 subdivisions by 90-night breach count per city
for city in CITIES:
    df = results[city]
    top = (
        df.loc[df['breach_90_flag']]
          .groupby('geo_key')
          .size()
          .sort_values(ascending=False)
          .head(10)
    )
    print(f'\nTop 10 subdivisions by 90-night breach count — {city.upper()}:')
    print(top.to_string())


Top 10 subdivisions by 90-night breach count — BARCELONA:
geo_key
la Dreta de l'Eixample                   95
la Sagrada Família                       53
el Poble-sec                             46
l'Antiga Esquerra de l'Eixample          43
la Vila de Gràcia                        34
Sant Pere, Santa Caterina i la Ribera    31
la Nova Esquerra de l'Eixample           31
el Raval                                 28
Sant Antoni                              26
el Poblenou                              21

Top 10 subdivisions by 90-night breach count — LONDON:
geo_key
Whitechapel         44
Paddington          42
Westbourne Green    41
Marylebone          35
Chelsea             26
North Kensington    26
Barnsbury           24
West Kensington     24
Earl's Court        23
Shoreditch          23


## 6. Handover note

Files written for Member 3 to pick up:

- `data/processed/barcelona/barcelona_listings_features.csv`
- `data/processed/london/london_listings_features.csv`

Schema is identical across both cities. Member 3 next steps:

1. Concatenate both files (`pd.concat`) for the cross-city analysis.
2. Use `geo_key` as the neighbourhood grouping column — falls back to borough/district when subdivision is missing.
3. The breach flags at 90 / 60 / 30 nights are pre-computed at listing level — sum them per `geo_key` for the policy simulation.
4. `host_commercial_tier`, `is_active_recent`, `host_revenue_total` are ready for clustering inputs.

All feature logic lives in `src/features.py` — import the functions there if you need to extend or re-run.